# 10주차 — RAG (1) 적재 · 분할 · 임베딩 (Colab판)

「최신인공지능」 2026 · 10주차 실습

| 실습 | 교시 | 내용 |
|------|------|------|
| 실습 1 | 1교시 | Loader 4종 — **무엇이 자동으로 붙는가** |
| 실습 2 ★★ | 2교시 | **같은 문서를 3가지로 잘라 검색 결과를 비교** |
| 3절 ★★ | 2교시 | 메타데이터 설계 — 11주차 전부를 좌우 |
| 실습 3 ★★ | 3교시 | 임베딩 + **인덱스 저장** (11·13주차가 이어 씀) |

> ### ⚠️⚠️ Colab 에서 반드시 알아야 할 것 — 파일이 사라집니다
>
> **Colab 런타임의 디스크는 세션이 끝나면 지워집니다.**
> 그런데 오늘 만드는 **인덱스는 11주차 실습 1과 13주차 실습 4가 그대로 이어 씁니다.**
>
> → 그래서 이 노트북은 **Google Drive 를 마운트**해 인덱스를 거기에 저장합니다.
>
> | | 실습실 (원본) | Colab (이 노트북) |
> |---|---|---|
> | 인덱스 위치 | `week10/code/index_recursive/` | `MyDrive/langchain-2026/week10/index_recursive/` |
> | 다음 주 사용 | `git clone` 후 그대로 | Drive 를 다시 마운트하면 그대로 ★ |
> | 백업 | `git commit` | Drive 자체가 백업 |
>
> ⚠️ **Drive 마운트를 건너뛰면 다음 주에 처음부터 다시 만들어야 합니다.**

## 0. 환경 준비

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Colab 환경 준비 — 매 세션 1회 실행 (재실행 안전)
# ══════════════════════════════════════════════════════════════
WEEK_MODELS   = ["chat", "embed"]
WEEK_PACKAGES = ("langchain langchain-core langchain-community langchain-ollama "
                 "langchain-text-splitters python-dotenv langsmith "
                 "pypdf beautifulsoup4 faiss-cpu numpy")
WEEK_SECRETS  = ["LANGSMITH_API_KEY"]

# ──────────────────────────────────────────────────────────────
import os, shutil, subprocess, sys, time, urllib.request

IN_COLAB = "google.colab" in sys.modules
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

GPU   = shutil.which("nvidia-smi") is not None and sh("nvidia-smi").returncode == 0
CHAT  = os.environ.setdefault("MODEL",       "gemma3:4b" if GPU else "gemma3:1b")
SMALL = os.environ.setdefault("SMALL_MODEL", "gemma3:1b")
EMBED = os.environ.setdefault("EMBED_MODEL", "nomic-embed-text")
TOOL  = os.environ.setdefault("TOOL_MODEL",  "qwen3:4b")
PICK  = {"chat": CHAT, "small": SMALL, "embed": EMBED, "tool": TOOL}

print(f"[1/6] 런타임   {'GPU 있음 ✅' if GPU else 'CPU 전용 ⚠️'}   →  대화 모델 {CHAT}")

print("[2/6] 패키지 설치 중… (faiss-cpu 가 있어 조금 걸립니다)")
r = sh(f"{sys.executable} -m pip install -q {WEEK_PACKAGES}")
print("       ✅ 완료" if r.returncode == 0 else "       ❌ 실패\n" + r.stderr[-600:])

if shutil.which("ollama") is None:
    print("[3/6] Ollama 설치 중… (약 30초)")
    sh("curl -fsSL https://ollama.com/install.sh | sh")
print("[3/6] Ollama  " + ("✅ 준비됨" if shutil.which("ollama") else "❌ 설치 실패"))

def alive():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not alive():
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if alive():
            break
        time.sleep(1)
print("[4/6] 서버    " + ("✅ 응답함" if alive() else "❌ 미응답 — 이 셀을 다시 실행하세요"))

have = {ln.split()[0] for ln in sh("ollama list").stdout.splitlines()[1:] if ln.strip()}
for key in WEEK_MODELS:
    name = PICK[key]
    if name in have:
        print(f"[5/6] {name:<20s} ✅ 이미 있음")
        continue
    print(f"[5/6] {name:<20s} ⏳ 내려받는 중…")
    t0 = time.time()
    r = sh(f"ollama pull {name}")
    print(f"       {'✅ 완료' if r.returncode == 0 else '❌ 실패'}  ({time.time() - t0:.0f}초)")

for k in WEEK_SECRETS:
    if not os.getenv(k) and IN_COLAB:
        try:
            from google.colab import userdata
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
os.environ.setdefault("LANGSMITH_PROJECT", "week10-rag")
os.environ.setdefault("USER_AGENT", "dongyang-ai-class/2026 (교육용 실습)")

# ── [6/6] Google Drive 마운트 — ★★ 인덱스를 다음 주까지 살려 둡니다 ──
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/langchain-2026/week10")
else:
    WORK = Path("./week10")          # 로컬 Jupyter 에서 열었을 때
WORK.mkdir(parents=True, exist_ok=True)

DATA      = WORK / "data"
INDEX_DIR = WORK / "index_recursive"   # ⚠️ 이 이름을 바꾸지 마십시오. 11·13주차가 씁니다 ★★
DATA.mkdir(exist_ok=True)

print(f"[6/6] 작업 폴더 {WORK}")
print(f"       인덱스   {INDEX_DIR}")

print("\n" + "=" * 62)
print(f"준비 완료 — MODEL='{CHAT}'  EMBED_MODEL='{EMBED}'")
print("=" * 62)

## 실습 자료 만들기

실습실에서는 `week10/code/data/` 에 있는 파일들입니다.
Colab 에는 저장소가 없으므로 **아래 셀이 같은 내용을 Drive 에 만들어 줍니다.**
(이미 있으면 덮어쓰지 않습니다)

> 🔶 **교수 준비물**: 실제 학칙 **PDF** 를 `MyDrive/langchain-2026/week10/data/학칙.pdf` 로
> 올려 두십시오. PDF 파싱의 난점(머리말·표·다단)은 **진짜 PDF 라야 보입니다.**

In [ ]:
DOC_MD = """# 제1장 총칙

## 제1조(목적)

이 학칙은 고등교육법 및 관계 법령에 따라 본 대학의 교육 목적을 달성하기 위하여
학사 운영에 필요한 사항을 규정함을 목적으로 한다.

## 제2조(적용 범위)

이 학칙은 본 대학에 재학 중인 모든 학생에게 적용한다. 다만 별도의 규정이 있는
경우에는 그 규정을 우선하여 적용한다.

## 제3조(학년도와 학기)

① 학년도는 3월 1일에 시작하여 다음 해 2월 말일에 끝난다. ② 학년도는 두 학기로
나누며, 각 학기의 수업일수는 15주 이상으로 한다. ③ 총장이 필요하다고 인정할 때에는
계절 수업을 개설할 수 있다. ④ 각 학기의 개시일과 종료일은 해당 학년도 학사일정으로
따로 정하며, 학사일정은 매 학년도가 시작되기 전에 학교 누리집에 공고한다.

# 제3장 휴학

## 제12조(일반휴학)

① 학생이 질병, 가사, 그 밖의 사유로 학업을 계속할 수 없을 때에는 총장의 허가를
받아 휴학할 수 있다. ② 일반휴학은 학기 단위로 신청하며 통산 6개 학기를 초과할 수
없다. ③ 휴학원은 해당 학기 수업일수 4분의 1이 지나기 전까지 제출하여야 한다.
④ 제2항의 기간을 산정할 때 군 복무 휴학과 임신·출산·육아 휴학은 포함하지 아니한다.

## 제13조(군 복무 휴학)

① 병역 의무를 이행하기 위한 휴학은 그 복무 기간을 휴학 기간으로 한다. ② 군 복무
휴학은 제12조 제2항의 통산 기간에 포함하지 아니한다. ③ 입영 통지서 사본을 첨부하여
휴학원을 제출하여야 하며, 부득이한 경우 사후에 제출할 수 있다.

## 제14조(질병 휴학)

① 질병으로 인한 휴학은 의료기관이 발행한 진단서를 첨부하여 신청한다. ② 진단서에
기재된 요양 기간이 30일 이상인 경우에 한하여 허가한다. ③ 질병 휴학은 제12조
제2항의 통산 기간에 포함한다.

## 제15조(임신·출산·육아 휴학)

① 임신·출산 또는 만 8세 이하 자녀의 양육을 위한 휴학을 신청할 수 있다.
② 이 휴학은 통산 4개 학기까지 허가하며, 제12조 제2항의 통산 기간에 포함하지 아니한다.

# 제4장 복학

## 제16조(복학)

① 휴학 기간이 만료된 학생은 그 다음 학기에 복학하여야 한다. ② 복학원은 해당 학기
개시 전 정해진 기간에 제출한다. ③ 휴학 기간이 만료되었음에도 복학하지 아니한
학생은 제20조에 따라 제적할 수 있다.

## 제17조(조기 복학)

① 휴학 사유가 소멸한 학생은 휴학 기간이 만료되기 전이라도 복학을 신청할 수 있다.
② 군 복무 휴학자의 조기 복학은 전역 예정일 기준으로 신청할 수 있다.

# 제5장 제적 및 재입학

## 제20조(제적)

① 다음 각 호의 어느 하나에 해당하는 학생은 제적한다. 1. 휴학 기간이 만료된 후
정해진 기간에 복학하지 아니한 자 2. 정해진 기간에 등록을 완료하지 아니한 자
3. 재학 연한을 초과한 자 4. 징계에 의하여 제적이 결정된 자

## 제21조(재입학)

① 제적된 자는 제적일로부터 3년 이내에 1회에 한하여 재입학을 신청할 수 있다.
② 징계에 의하여 제적된 자는 재입학을 허가하지 아니한다.

# 제6장 수업 및 학점

## 제25조(수업 연한과 재학 연한)

① 수업 연한은 3년으로 한다. ② 재학 연한은 수업 연한의 2배를 초과할 수 없으며,
휴학 기간은 재학 연한에 산입하지 아니한다.

## 제26조(학점 이수)

① 졸업에 필요한 학점은 120학점 이상으로 한다. ② 한 학기에 신청할 수 있는 학점은
21학점을 초과할 수 없다. 다만 직전 학기 성적이 평점 평균 3.5 이상인 학생은
24학점까지 신청할 수 있다.

## 제27조(계절 수업)

① 계절 수업으로 취득할 수 있는 학점은 매 계절 6학점을 초과할 수 없다.
② 계절 수업의 수업 시간은 학점당 15시간 이상으로 한다.

# 제7장 성적

## 제30조(성적 평가)

① 성적은 출석, 과제, 시험 등을 종합하여 평가한다. ② 수업 시간의 4분의 1을 초과하여
결석한 학생에게는 F 학점을 부여한다.

## 제31조(재수강)

① C+ 이하의 성적을 취득한 교과목은 재수강할 수 있다. ② 재수강한 교과목의 성적은
A0를 초과하여 부여할 수 없다.

# 제8장 졸업

## 제35조(졸업 요건)

① 수업 연한을 이수하고 제26조의 졸업 학점을 취득한 자에게 졸업을 인정한다.
② 학과가 정한 졸업 인증 요건을 충족하여야 한다.

## 제36조(학위 수여)

① 졸업이 인정된 자에게는 전문학사 학위를 수여한다.
"""

DOC_TXT = """동양미래대학교 학사 안내 (요약본)

휴학
학생이 질병, 가사, 그 밖의 사유로 학업을 계속할 수 없을 때에는 총장의 허가를 받아
휴학할 수 있습니다. 일반휴학은 학기 단위로 신청하며 통산 6개 학기를 초과할 수 없습니다.
군 복무 휴학과 임신·출산·육아 휴학은 이 통산 기간에 포함되지 않습니다.

복학
휴학 기간이 만료된 학생은 그 다음 학기에 복학하여야 합니다. 휴학 사유가 소멸한 경우에는
휴학 기간이 만료되기 전이라도 조기 복학을 신청할 수 있습니다.

등록
등록은 정해진 기간에 완료하여야 하며, 기간 내에 등록하지 않은 학생은 제적될 수 있습니다.

수강신청
한 학기에 신청할 수 있는 학점은 21학점을 초과할 수 없습니다. 다만 직전 학기 성적이
평점 평균 3.5 이상인 학생은 24학점까지 신청할 수 있습니다.

성적
수업 시간의 4분의 1을 초과하여 결석한 학생에게는 F 학점을 부여합니다.
C+ 이하의 성적을 취득한 교과목은 재수강할 수 있으며, 재수강 성적은 A0를 초과할 수 없습니다.

졸업
졸업에 필요한 학점은 120학점 이상이며, 학과가 정한 졸업 인증 요건을 충족하여야 합니다.
"""

DOC_CSV = """번호,분류,질문,답변
1,휴학,일반휴학은 최대 몇 학기까지 할 수 있나요?,일반휴학은 학기 단위로 신청하며 통산 6개 학기를 초과할 수 없습니다.
2,휴학,군 복무 휴학도 통산 기간에 포함되나요?,포함되지 않습니다. 군 복무 휴학은 일반휴학 통산 기간과 별도로 산정합니다.
3,휴학,휴학원은 언제까지 내야 하나요?,해당 학기 수업일수 4분의 1이 지나기 전까지 제출하여야 합니다.
4,휴학,질병으로 휴학하려면 무엇이 필요한가요?,의료기관이 발행한 진단서가 필요하며 요양 기간이 30일 이상이어야 합니다.
5,복학,휴학 기간이 끝나면 반드시 복학해야 하나요?,네. 복학하지 않으면 제적될 수 있습니다.
6,복학,휴학 기간이 남았는데 미리 돌아올 수 있나요?,휴학 사유가 소멸하면 조기 복학을 신청할 수 있습니다.
7,수강,한 학기에 최대 몇 학점까지 신청할 수 있나요?,21학점까지이며 직전 학기 평점 평균 3.5 이상이면 24학점까지 가능합니다.
8,성적,결석을 몇 번 하면 F인가요?,수업 시간의 4분의 1을 초과하여 결석하면 F 학점이 부여됩니다.
9,성적,재수강하면 A+를 받을 수 있나요?,받을 수 없습니다. 재수강 성적은 A0를 초과하여 부여하지 않습니다.
10,졸업,졸업 학점은 몇 학점인가요?,120학점 이상이며 학과가 정한 졸업 인증 요건도 충족해야 합니다.
"""

for name, body in [("학칙.md", DOC_MD), ("규정.txt", DOC_TXT), ("faq.csv", DOC_CSV)]:
    path = DATA / name
    if path.exists():
        print(f"  [i] {name:<12s} 이미 있음 — 그대로 둡니다")
    else:
        path.write_text(body, encoding="utf-8")
        print(f"  [+] {name:<12s} 생성 ({len(body)}자)")

PDF_PATH = DATA / "학칙.pdf"
print(f"\n  PDF: {'✅ 있음' if PDF_PATH.exists() else '⬜ 없음 — 실습 1의 PDF 절은 건너뜁니다 🔶'}")
print(f"  파일 목록: {sorted(f.name for f in DATA.iterdir())}")

## 실습 1 (1교시) — Loader 4종

**Loader = 무엇이든 읽어서 `Document` 객체 리스트로 만든다.**

```python
Document(
    page_content="문서의 본문 텍스트...",
    metadata={"source": "학칙.pdf", "page": 7},      # ★ 이게 오늘의 복선
)
```

> ★ **`metadata` 를 지금 기억해 두십시오.**
> 2교시 3절(메타데이터 설계)과 **11주차(필터 검색·출처 표기)** 의 근거가 됩니다.

**Loader 별 '단위' 가 다릅니다** ★

| Loader | Document 개수 | 자동 metadata |
|---|---|---|
| Text | 1개 (파일 전체) | `source` |
| PDF | **페이지 수만큼** ★ | `source`, **`page`** ★ |
| CSV | **행 수만큼** ★ | `source`, `row` |
| Web | 1개 | `source`, `title` |

> 📌 **"쓰레기를 넣으면 쓰레기가 나옵니다."**
> 파이프라인 6단계 중 ①에서 망가진 것은 ⑥에서 **절대 복구되지 않습니다.**

In [ ]:
from langchain_community.document_loaders import CSVLoader, TextLoader, WebBaseLoader

# 🔶 학교 공지 등 실제로 접근 가능한 주소로 바꾸십시오.
WEB_URL = "https://example.com"
LOAD_WEB = False        # True 로 바꾸면 Web Loader 까지 실행합니다


def show(name: str, docs) -> None:
    print("=" * 60)
    if not docs:
        print(f"[{name}]  건너뜀")
        return
    print(f"[{name}]  Document 개수: {len(docs)}")
    print("metadata:", docs[0].metadata)          # ★ 무엇이 '자동으로' 붙는가
    print("본문 앞 200자:")
    print(docs[0].page_content[:200].strip())
    print()


# ① 텍스트 — 파일 전체가 Document 1개. 구조 정보가 전혀 없다.
show("Text", TextLoader(str(DATA / "규정.txt"), encoding="utf-8").load())

# ② PDF — 페이지 단위로 쪼개진다. metadata 에 page 가 붙는다 ★
if PDF_PATH.exists():
    from langchain_community.document_loaders import PyPDFLoader
    show("PDF", PyPDFLoader(str(PDF_PATH)).load())
else:
    print("=" * 60)
    print(f"""[PDF]  건너뜀 — {PDF_PATH.name} 가 없습니다. 🔶

  ⚠️ 이 절의 핵심(파싱 난점)은 **진짜 PDF** 라야 보입니다.
     교수 준비물: 실제 학칙 PDF 를 Drive 의 data/학칙.pdf 로 올려 두십시오.

     ┌─────────────────────────────────┐
     │  동양미래대학교 학칙        - 7 -│  ← 머리말·페이지 번호가 본문에 섞인다
     ├─────────────────────────────────┤
     │  제3장 휴학                      │
     │  ┌──────┬──────┬──────┐         │
     │  │ 구분  │ 기간  │ 비고  │        │  ← 표가 한 줄 텍스트로 뭉개진다 ⚠️
     │  └──────┴──────┴──────┘         │
     │  ① 일반휴학은 ...                │  ← 단이 나뉘면 순서가 뒤섞인다
     └─────────────────────────────────┘
""")

# ③ CSV — 행 단위로 Document 가 만들어진다. 열 이름이 본문에 섞인다 ★
show("CSV", CSVLoader(str(DATA / "faq.csv"), encoding="utf-8").load())

# ④ 웹 — 메뉴·광고·푸터가 본문에 섞인다 ⚠️
if LOAD_WEB:
    try:
        show("Web", WebBaseLoader(WEB_URL).load())
    except Exception as e:
        print(f"[Web]  건너뜀 — {type(e).__name__}: {str(e)[:60]} 🔶")
else:
    print("=" * 60)
    print("[Web]  건너뜀 — 켜려면 위 셀의 LOAD_WEB = True\n")

### 파싱 난점 — 무엇을 관찰할 것인가 ★

| Loader | Document 개수 | 자동 metadata | 난점 |
|---|---|---|---|
| Text | 1개 (파일 전체) | `source` | 구조 정보가 전혀 없음 |
| PDF | 페이지 수만큼 | `source`, `page` ★ | 머리말·표·다단·스캔본 ⚠️ |
| CSV | 행 수만큼 | `source`, `row` | 열 이름이 본문에 섞임 |
| Web | 1개 | `source`, `title` | 메뉴·광고·푸터가 섞임 ⚠️ |

**⚠️ PDF 가 가장 까다롭습니다**

| 문제 | 결과 | 대응 |
|---|---|---|
| 머리말·페이지 번호 | 모든 청크에 잡음이 섞임 | 전처리로 제거 (정규식) |
| 표 | 행·열 관계가 사라짐 | 표 특화 파서 / 수동 정리 |
| 다단 레이아웃 | 읽는 순서가 뒤바뀜 | 레이아웃 인식 파서 |
| 스캔 PDF (이미지) | 텍스트가 아예 없음 ⚠️ | OCR (본 과목 범위 밖) |

> 📌 **"쓰레기를 넣으면 쓰레기가 나옵니다."**
> RAG 프로젝트 시간의 상당 부분이 ① Load 와 ② Split 에 들어갑니다.
>
> 💡 **9주차 연결**: `WebBaseLoader` 로 가져온 페이지에 주입 문장이 있다면?
> 그대로 청크가 되어 **벡터 저장소에 들어갑니다. 한 번 들어가면 계속 검색됩니다** ⚠️
> → **"가져온 텍스트는 데이터다. 지시가 아니다."** ★

## 실습 2 ★★ (2교시) — 같은 문서를 3가지로 자른다

> ### ★ 이 교시가 10주차의 핵심입니다
>
> 학생들은 보통 *"임베딩 모델이 좋으면 검색이 잘 되겠지"* 라고 생각합니다.
> 실제로는 **어떻게 잘랐는가가 검색 품질의 상한을 결정합니다.**

| | 방식 | 특징 |
|---|---|---|
| **A** | 고정 크기 | 단순·빠름 ⚠️ **문장·단어 중간에서 끊긴다** |
| **B** | 재귀적 | 큰 경계부터 시도 ★ **실무 기본값** |
| **C** | 구조 기반 | 섹션 통째로 유지 ★★ **메타데이터가 자동 생성** |

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    CharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

DOC_PATH = DATA / "학칙.md"
CHUNK_SIZE, CHUNK_OVERLAP = 500, 50

# ★ 한국어 구분자를 추가했습니다.
#   기본 separators 는 영어 기준(". ")이라 한국어 문장 끝("다. ")을 못 잡습니다.
KO_SEPARATORS = ["\n\n", "\n", "다. ", ". ", " ", ""]
HEADERS = [("#", "장"), ("##", "조")]

# 🔶 사전 확인 필수: 이 질문으로 세 인덱스의 결과가 실제로 갈리는지 미리 돌려 보십시오 ★
QUESTION = "일반휴학은 통산 최대 몇 학기까지 할 수 있나요?"

raw = TextLoader(str(DOC_PATH), encoding="utf-8").load()[0].page_content
print(f"문서: {DOC_PATH.name}  ({len(raw)}자)\n")


# ── A. 고정 크기 ────────────────────────────────────────────
def split_a(raw: str) -> list[str]:
    """500자에서 기계적으로 끊는다. ⚠️ 이 방식의 실패가 오늘 실습의 출발점."""
    return CharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=0, separator=""
    ).split_text(raw)


# ── B. 재귀적 ──────────────────────────────────────────────
def split_b(raw: str) -> list[str]:
    """가능한 한 큰 경계를 지키면서 500자를 맞춘다. ★ 실무 기본값

       ① "\\n\\n"(문단)으로 나눠본다  → 500자 이하가 되면 채택
       ② 안 되면 "\\n"(줄)로
       ③ 안 되면 "다. "·". "(문장)으로
       ④ 그래도 안 되면 " "(단어) → 마지막엔 글자 단위
    """
    return RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, separators=KO_SEPARATORS
    ).split_text(raw)


# ── C. 구조 기반 ───────────────────────────────────────────
def split_c(raw: str) -> list[Document]:
    """섹션(조)이 통째로 유지되고, ★★ metadata 가 자동으로 붙는다."""
    return MarkdownHeaderTextSplitter(headers_to_split_on=HEADERS).split_text(raw)


a, b = split_a(raw), split_b(raw)
c_docs = split_c(raw)
c = [d.page_content for d in c_docs]

for name, chunks in [("A 고정", a), ("B 재귀", b), ("C 구조", c)]:
    lens = [len(x) for x in chunks]
    print("=" * 60)
    print(f"[{name}] {len(chunks)}개  평균 {sum(lens) // len(lens)}자  "
          f"최소 {min(lens)}  최대 {max(lens)}")
    print("--- 첫 청크 끝부분 80자 (어디서 끊겼는지 보기) ★ ---")
    print("..." + chunks[0][-80:].replace("\n", " "))

print("=" * 60)
print("""
  [A 고정] ...신청하며 통산            ← ⚠️ 숫자 바로 앞에서 끊겼다
  [B 재귀] ...초과할 수 없다.          ← ✅ 문장 끝
  [C 구조] (제12조 전체가 한 청크)      ← ✅ 조 단위
""")

# C 의 metadata 가 자동 생성되는 것을 보여준다 ★★
print("  ★★ C 는 metadata 가 자동으로 붙습니다 — 3절의 근거입니다")
for d in c_docs[3:5]:
    print(f"    {d.metadata}")

GROUPS = {"A 고정": a, "B 재귀": b, "C 구조": c}

> ### ⚠️ A 에서 무슨 일이 일어났는지 보십시오
>
> `"일반휴학은 학기 단위로 신청하며 통산"` 에서 끊긴 청크가 만들어졌습니다.
> 이 청크는 질문(*"일반휴학은 통산 몇 학기?"*)과 매우 비슷해서 **검색은 1위로 잘 됩니다.**
>
> 그런데 **몇 학기인지가 그 안에 없습니다.** ★★
>
> → **검색은 성공했는데 답을 못 만듭니다.** 그리고 모델은 그럴듯하게 지어냅니다.

In [ ]:
# ── 세 인덱스에 같은 질문을 던져 무엇이 나오는지 본다 ★★ ──
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

EMBED_MODEL = os.environ["EMBED_MODEL"]
emb = OllamaEmbeddings(model=EMBED_MODEL)

print(f"질문: {QUESTION}\n")
for name, chunks in GROUPS.items():
    store = FAISS.from_texts(chunks, emb)
    hits = store.similarity_search(QUESTION, k=2)
    print("=" * 60)
    print(f"[{name}]")
    for i, d in enumerate(hits, 1):
        text = d.page_content.replace("\n", " ")
        # ★ 답(숫자)이 이 청크 안에 실제로 들어 있는가 — 이것이 판정 기준
        mark = "✅ 답 포함" if "6개 학기" in d.page_content else "❌ 답 없음"
        print(f"  {i}. [{mark}] {text[:110]}...")

### 학생이 채울 표

| 분할 | 청크 수 | 평균 길이 | 검색 1위 청크에 **답이 들어 있는가** |
|---|---|---|---|
| A 고정 | | | |
| B 재귀 | | | |
| C 구조 | | | |

### 무엇을 읽어낼 것인가 ★

| 관찰 | 의미 |
|---|---|
| A 에서 답이 잘린 청크가 1위 | **검색은 성공했는데 답을 못 만든다** ★★ |
| B 는 문장이 온전함 | 같은 크기인데 **쓸 수 있는 청크**가 된다 |
| C 는 조 단위로 완결 | 규정·매뉴얼처럼 구조가 있는 문서에 최적 |
| C 의 청크 길이 편차가 큼 | 긴 섹션은 다시 잘라야 함 (2단 구성) |

> ### 📌 "검색 품질의 상한은 임베딩 모델이 아니라 분할이 정합니다."
>
> 같은 임베딩 모델, 같은 질문인데 결과가 달랐습니다. **바꾼 것은 자르는 방법뿐입니다.**
>
> 🔶 결과가 안 갈리면: `CHUNK_SIZE` 를 더 작게 잡거나, 경계에 걸치는 질문으로 바꾸십시오.
> **갈리지 않으면 이 실습의 의미가 사라집니다.** ★

## 2교시 3절 ★★ — 메타데이터 설계

```
[10주차 — 지금]                  [11주차 — 나중]
청크에 metadata 를 심는다   →    · 필터 검색  "2026년 개정본에서만"
  source / page / 장 / 조         · Self-Query "3장에서만 찾아줘"
  year / category                 · ★ 출처 표기 "[학칙 3장 12조]"
                                    │
⚠️ 안 심으면?                →     └ 만들 수 없습니다. 정보가 없으니까 ★★
```

> ★ **실무의 2단 구성**을 씁니다: C(구조)로 크게 나눈 뒤 → 긴 섹션만 B(재귀)로 다시.
> - 구조로 나누면 **장·조 metadata 가 공짜로** 생깁니다
> - 재귀로 다듬으면 **청크 길이 편차**가 잡힙니다

In [ ]:
def build_chunks_with_metadata(raw_text: str | None = None) -> list[Document]:
    """분할 단계에서 메타데이터를 심는다. ★★ 이 절이 11주차 전부를 좌우합니다."""
    raw_text = raw_text if raw_text is not None else raw

    sections = split_c(raw_text)          # 장·조 metadata 가 붙은 섹션들
    sub = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, separators=KO_SEPARATORS
    )

    chunks: list[Document] = []
    for sec in sections:
        for piece in sub.split_text(sec.page_content):
            chunks.append(
                Document(
                    page_content=piece,
                    metadata={
                        # ── 출처 표기용 ★ (11주차 3교시 Citation) ────────
                        "source": DOC_PATH.name,
                        "chapter": sec.metadata.get("장", ""),
                        "article": sec.metadata.get("조", ""),
                        # ⚠️ page 는 PDF Loader 라야 붙습니다. .md 에는 아예 없습니다.
                        #    "나중에 복원할 수 없는 것" 의 대표 사례입니다 ★
                        #    → 11주차 출처 표기에서 p.? 로 표시됩니다.
                        #    ⚠️ 값이 없다고 None 을 넣지 마십시오. 일부 벡터 저장소
                        #       (Chroma 등)는 metadata 에 None 을 거부합니다. 🔶
                        #       "없는 키" 로 두는 편이 안전합니다.
                        # ── 필터 검색용 ★ (11주차 1교시) ────────────────
                        # ⚠️ 타입을 맞추십시오. 2026(숫자)과 "2026"(문자)은
                        #    범위 필터에서 다르게 동작합니다.
                        "year": 2026,
                        "category": "학사",
                        "doc_type": "규정",
                        # ── 운영·디버깅용 ────────────────────────────────
                        "chunk_id": len(chunks),
                        # ★ 어떻게 잘랐는지 기록해 두면 7주차 방식으로
                        #   분할 전략을 A/B 비교할 수 있습니다.
                        "splitter": f"markdown→recursive-{CHUNK_SIZE}-{CHUNK_OVERLAP}",
                    },
                )
            )
    return chunks


chunks = build_chunks_with_metadata()
print(f"메타데이터를 심은 청크 {len(chunks)}개\n")
for d in chunks[:3]:
    print("─" * 60)
    print(d.metadata)
    print(d.page_content[:90].replace("\n", " "), "...")

### 설계 원칙 3가지 ★

| 원칙 | 이유 |
|---|---|
| **나중에 복원할 수 없는 것을 우선** | 페이지 번호·섹션 제목은 자르고 나면 사라진다 ★ |
| **필터로 쓸 값은 타입을 맞춘다** | `year: 2026`(숫자) vs `"2026"`(문자) — 범위 필터가 달라짐 |
| 본문에 이미 있는 것은 안 넣는다 | 중복은 저장 비용만 늘림 |

> ⚠️ **과하게도, 부족하게도 넣지 마십시오.** 기준은 하나입니다 —
> ***"11주차에 이걸로 무엇을 하고 싶은가?"***
>
> 📌 **학생 활동**: "내 미니 프로젝트 문서에는 어떤 메타데이터가 필요할까?" 를
> `CONTEXT.md` 에 한 줄이라도 적으십시오. ★

## 실습 3 ★★ (3교시) — 임베딩과 인덱스 저장

🎯 오늘은 **"고르고 붙이는 것"** 만 합니다. 임베딩 원리는 2주차에 이미 배웠습니다.
다만 하나는 반드시 확인합니다 — **이 모델이 한국어를 제대로 다루는가?** ★★

### 임베딩 모델 선택 기준 4가지 ★

| 기준 | 내용 |
|---|---|
| **한국어 지원** ★★ | 한국어를 학습한 모델인가 — 영어 전용은 한국어 유사도가 무너진다 |
| 차원 | 384 / 768 / 1024 … 크면 표현력↑ 저장·검색 비용↑ |
| 속도 | 초당 처리 청크 수 — 문서 1만 개면 차이가 큽니다 |
| 로컬 실행 가능 | VRAM 요구량 — 8GB 안에 생성 LLM과 함께 올라가는가 ★ |

```
⚠️ 8GB VRAM 예산 (11주차 기준)
   생성 LLM (8B Q4)     ≒ 5.0 GB
   임베딩 모델          ≒ 0.3 GB
   KV 캐시 + 오버헤드   ≒ 1.5 GB
   ─────────────────────────────
   합계                 ≒ 6.8 GB   → 여유 있음 ✅
```

In [ ]:
# ── ① 한 문장 임베딩 ────────────────────────────
v = emb.embed_query("일반휴학은 최대 몇 학기까지 가능한가?")
print("── ① 한 문장 임베딩 ────────────────────────────")
print("  차원   :", len(v))
print("  앞 5개 :", [round(x, 4) for x in v[:5]])

print("""
  ★ embed_query 와 embed_documents 를 구분하십시오.
    일부 임베딩 모델은 **질문과 문서를 다르게 처리**합니다(접두어를 붙이는 등).
    그래서 메서드가 나뉘어 있습니다. 섞어 쓰면 검색 품질이 떨어질 수 있습니다.
""")

### ②-2. 이 모델이 한국어 '의미' 를 잡습니까? ★★

2주차에 배운 **코사인 유사도가 여기서 검증 도구로** 쓰입니다.

> ### ★★ 판정 기준은 '점수의 절대값' 이 아닙니다
>
> **의미가 같은 쌍이 의미가 다른 쌍보다 확실히 높은가** 입니다.
> (임베딩 모델마다 점수 분포가 달라서 절대값은 비교 기준이 못 됩니다)

In [ ]:
import numpy as np


def cos(a, b) -> float:
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


# (질문, 비교문, 설명, 같은 의미인가)
PAIRS = [
    ("휴학 절차", "학교를 잠시 쉬는 방법", "높아야 함 ★", True),
    ("휴학 절차", "졸업 요건", "낮아야 함 ★", False),
    ("휴학 절차", "휴학 신청 방법", "매우 높아야 함", True),
    ("졸업 학점", "졸업에 필요한 학점 수", "높아야 함", True),
    ("졸업 학점", "군 복무 휴학 신청", "낮아야 함", False),
]


def korean_check(embeddings, model_name: str = "") -> bool:
    print(f"── 한국어 검증 ({model_name or EMBED_MODEL}) ★★ ──────────")
    same, diff = [], []
    for a_, b_, expect, is_same in PAIRS:
        s = cos(embeddings.embed_query(a_), embeddings.embed_query(b_))
        (same if is_same else diff).append(s)
        print(f"  {s:.3f}  {a_} ↔ {b_}   ({expect})")

    # ★ 핵심 지표: '의미가 같은 쌍' 의 최저점 − '의미가 다른 쌍' 의 최고점
    #   이 값이 0 이하면 두 무리가 **겹칩니다** = 의미를 구분하지 못합니다.
    margin = min(same) - max(diff)
    print(f"\n  같은 의미 최저 {min(same):.3f}  ↔  다른 의미 최고 {max(diff):.3f}")
    print(f"  ★ 판정 지표(margin) = {margin:+.3f}")

    if margin > 0.03:
        print("\n  ✅ 의미가 같은 쌍이 확실히 높습니다. 이 모델을 써도 됩니다.\n")
        return True

    print(f"""
  ⚠️⚠️ **이 모델은 한국어 의미를 구분하지 못합니다.** (margin {margin:+.3f} ≤ 0.03)

     "휴학 절차" 와 "졸업 요건"(전혀 다른 내용)의 점수가
     "휴학 절차" 와 "학교를 잠시 쉬는 방법"(같은 내용)만큼 높습니다.
     **점수가 높은 것이 문제가 아니라, 구분이 안 되는 것이 문제입니다.** ★★

     ⚠️ 그대로 두면 11주차에 검색이 안 되는데 **원인을 못 찾게 됩니다.**

  🔶 **교체를 검토하십시오.** 한국어를 학습한 임베딩 모델 후보:
        !ollama pull bge-m3                 # 다국어(한국어 포함) · 1024차원 · ~1.2GB
        !ollama pull qwen3-embedding:0.6b   # 다국어 · 소형
     그리고 os.environ["EMBED_MODEL"] 을 바꾼 뒤 이 검증을 **다시** 돌리십시오.
""")
    return False


KOREAN_OK = korean_check(emb)

In [ ]:
# ── 후보 모델을 나란히 비교하고 싶다면 (선택) ──
#    ⚠️ 없는 모델은 먼저 pull 이 필요합니다. 시간이 걸립니다.
CANDIDATES = []          # 예: ["nomic-embed-text", "bge-m3"]

results = {}
for name in CANDIDATES:
    try:
        results[name] = korean_check(OllamaEmbeddings(model=name), name)
    except Exception as e:
        print(f"── {name}: ❌ 사용 불가 ({type(e).__name__}) — ollama pull 이 필요합니다 🔶\n")
        results[name] = False

if len(results) > 1:
    print("=" * 60)
    for name, ok in results.items():
        print(f"  {'✅' if ok else '❌'}  {name}")
    print("\n  ✅ 인 모델을 EMBED_MODEL 로 지정하십시오. ★")

### ③ 산출물 저장 — 11·13주차가 그대로 이어 씁니다 ★★

> ### ⚠️⚠️ 마지막 단계를 반드시 하십시오
>
> **11주차 실습 1**과 **13주차 실습 4**가 오늘 만든 인덱스를 그대로 씁니다.
> 없으면 다음 주 도입부를 재적재로 날립니다.
>
> Colab 에서는 **Drive 에 저장하는 것이 곧 커밋**입니다.

#### 🔶 참고: 실습실에서 실제로 터지는 문제 — 한글 경로

FAISS 의 `save_local()` / `load_local()` 은 내부적으로 C++ 함수를 호출합니다.
이 함수들은 **경로에 ASCII 가 아닌 문자(한글 등)가 있으면 실패합니다.**

```
RuntimeError: Error in FileIOWriter... could not open
    ...\최신인공지능\...\index.faiss for writing: Illegal byte sequence
```

학생 저장소가 `C:\Users\홍길동\바탕화면\수업\...` 같은 경로에 있으면
**10주차 마지막 단계에서 그대로 멈추고, 11·13주차가 연쇄로 막힙니다.** ⚠️

> 💡 **Colab 은 `/content/drive/MyDrive/...` 라 이 문제가 없습니다.**
> 아래 `save_index()` 는 실습실 대비 우회 코드를 그대로 담고 있습니다 —
> **실습실 PC 에서 돌릴 때를 위한 것입니다.** ★

In [ ]:
BYTES_NAME = "index.bytes"      # ② 대체 저장 형식


def path_is_ascii(path) -> bool:
    """이 경로에서 FAISS 의 C++ 저장이 될 것인가."""
    return str(Path(path).resolve()).isascii()


def save_index(store, directory) -> str:
    """인덱스를 저장하고, 어떤 방식으로 저장했는지 돌려준다."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    try:
        store.save_local(str(directory))          # ① 표준 API — 강의안 그대로 ★
        return "save_local"
    except Exception as e:                        # ⚠️ 한글 경로 등
        (directory / BYTES_NAME).write_bytes(store.serialize_to_bytes())   # ②
        print(f"  🔶 save_local() 이 실패해 바이트 방식으로 저장했습니다. ({type(e).__name__})")
        return "bytes"


def load_index(directory, embeddings):
    """저장 방식을 자동으로 판별해 읽는다. 없으면 None."""
    directory = Path(directory)
    if (directory / "index.faiss").exists():
        # allow_dangerous_deserialization: 내가 만든 로컬 파일이므로 신뢰합니다. 🔶
        # ⚠️ 남이 준 인덱스 파일에는 켜지 마십시오 — pickle 을 읽습니다.
        return FAISS.load_local(str(directory), embeddings,
                                allow_dangerous_deserialization=True)
    if (directory / BYTES_NAME).exists():
        return FAISS.deserialize_from_bytes(
            embeddings=embeddings,
            serialized=(directory / BYTES_NAME).read_bytes(),
            allow_dangerous_deserialization=True,
        )
    return None


print(f"현재 경로: {INDEX_DIR}")
print(f"경로가 ASCII 인가: {'✅ 예 — 표준 save_local 로 저장됩니다' if path_is_ascii(INDEX_DIR) else '❌ 아니오 — 바이트 방식으로 우회합니다'}")

In [ ]:
# ⚠️ chunks 는 3절에서 만든 'Document 리스트' 입니다.
#    split_text() 가 돌려준 '문자열 리스트' 가 아닙니다 —
#    메타데이터가 붙어 있어야 11주차 필터 검색·출처 표기가 가능합니다. ★
texts = [c.page_content for c in chunks]

print("── ② 청크 여러 개를 한 번에 ★ ──────────────────")
vectors = emb.embed_documents(texts)
print(f"  {len(vectors)}개 청크 → {len(vectors[0])}차원 벡터\n")

store = FAISS.from_documents(chunks, emb)      # ★ 메타데이터까지 함께 들어간다
how = save_index(store, INDEX_DIR)

print(f"✅ 인덱스 저장: {INDEX_DIR}  (방식: {how})")
print(f"   파일: {sorted(f.name for f in INDEX_DIR.iterdir())}")

# 저장한 것이 실제로 읽히는지 그 자리에서 확인 ★
reloaded = load_index(INDEX_DIR, emb)
hit = reloaded.similarity_search("일반휴학은 몇 학기까지 가능한가?", k=1)[0]
print(f"   되읽기 확인: {hit.metadata.get('article')} | "
      f"{hit.page_content[:50].replace(chr(10), ' ')}...")

print(f"""
⚠️⚠️ 이 인덱스는 Drive 에 있습니다 — 다음 주에 그대로 씁니다.

   경로: {INDEX_DIR}

   **11주차 실습 1**과 **13주차 실습 4**가 이 인덱스를 이어 씁니다.
   Drive 를 다시 마운트하면 그대로 남아 있습니다. 지우지 마십시오. ★★

   💡 실습실 저장소에도 함께 커밋해 두면 더 안전합니다:
       git add index_recursive
       git commit -m "week10: 인덱스 저장"
""")

## 오늘 확인할 것

- [ ] Loader 4종의 **Document 개수와 자동 metadata** 차이를 확인했다
- [ ] A 고정 분할이 **답 바로 앞에서 끊는** 것을 눈으로 봤다 ★★
- [ ] 세 인덱스에 같은 질문을 던져 **답 포함 여부**를 비교했다 ★★
- [ ] 청크에 **메타데이터를 심었다** (11주차 전부의 근거) ★★
- [ ] 임베딩 모델의 **한국어 margin** 을 검증했다 ★★
- [ ] **인덱스를 Drive 에 저장**하고 되읽기까지 확인했다 ★★

### 📌 다음 주 준비

**11주차 첫 셀에서 Drive 를 마운트하면 오늘의 인덱스가 그대로 로드됩니다.**
Drive 의 `langchain-2026/week10/index_recursive/` 폴더를 **지우지 마십시오.**

### 오늘의 한 줄

> **"검색 품질의 상한은 임베딩 모델이 아니라 분할이 정합니다."**